## 02 — MSD Data Preprocessing


In [1]:
import os
import math
import glob
import shutil
import h5py
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, lower, trim, concat_ws, count
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import when, row_number
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.functions import vector_to_array

## THE NEXT TWO CELLS ARE CRUCIAL FOR RUNNING HADOOP AND PYSPARK ON MY DEVICE


In [2]:
import sys

# On Windows, the system-level SPARK_HOME may point to a different Spark version.
# We override it here to make sure PySpark always finds its own bundled JARs.
os.environ["SPARK_HOME"]  = os.path.dirname(pyspark.__file__)

# Spark needs Hadoop's winutils.exe on Windows to handle local file operations
# (directory creation, temp files, etc.). We installed winutils at C:\hadoop\bin\.
os.environ["HADOOP_HOME"] = r"C:\hadoop"

# Prepend C:\hadoop\bin to PATH so the JVM can find hadoop.dll via
# System.loadLibrary("hadoop"). HADOOP_HOME alone only locates winutils.exe
# for Hadoop's shell helpers — it does NOT add the directory to
# java.library.path. Without this, NativeIO$Windows.access0 throws
# UnsatisfiedLinkError on Spark write operations.
hadoop_bin = r"C:\hadoop\bin"
if hadoop_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = hadoop_bin + os.pathsep + os.environ.get("PATH", "")

# Pin the Python interpreter for both the driver and the executor-side workers
# to the exact Python running this notebook. Without these, PySpark on Windows
# can pick a different Python for workers, causing them to die with
# "Connection reset" the moment Spark tries to deserialize Python rows.
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("SPARK_HOME    :", os.environ["SPARK_HOME"])
print("HADOOP_HOME   :", os.environ["HADOOP_HOME"])
print("PATH[0]       :", os.environ["PATH"].split(os.pathsep)[0])
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])

SPARK_HOME    : c:\Users\Mariam\anaconda3\envs\spotify-recommender\Lib\site-packages\pyspark
HADOOP_HOME   : C:\hadoop
PATH[0]       : C:\hadoop\bin
PYSPARK_PYTHON: c:\Users\Mariam\anaconda3\envs\spotify-recommender\python.exe


In [3]:
# local[*]  — uses all CPU cores on this machine (pseudo-distributed mode).
# spark.driver.memory — heap given to the JVM driver process.
# spark.local.dir     — MUST be on the same drive as the output directory (D:).
#   Spark writes temp files here, then renames them to the final output path.
#   Java's File.renameTo() cannot rename across different drives on Windows,
#   so if temp is on C: and output is on D:, every write fails silently.
# spark.hadoop.home.dir — tells the JVM where winutils lives at runtime.
# RawLocalFileSystem  — skips Unix chmod/chown calls that crash on Windows.
# fileoutputcommitter.algorithm.version=2 — avoids the rename-based commit phase.
# marksuccessfuljobs=false — skips writing the _SUCCESS marker file.

SPARK_TEMP_DIR = "D:/tmp/spark"
os.makedirs(SPARK_TEMP_DIR, exist_ok=True)

spark = SparkSession.builder \
    .appName("SpotifyPreprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.local.dir", SPARK_TEMP_DIR) \
    .config("spark.hadoop.home.dir", r"C:\hadoop") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .config("spark.hadoop.fs.file.impl.disable.cache", "true") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


def spark_write_csv(df, relative_path):
    """
    Write a Spark DataFrame to a single CSV file on Windows.
    Deletes the output directory with Python first so Spark never has to
    rename or overwrite an existing folder across drives.
    """
    if os.path.exists(relative_path):
        shutil.rmtree(relative_path)
    abs_path = os.path.abspath(relative_path).replace("\\", "/")
    df.coalesce(1).write.csv(f"file:///{abs_path}", header=True)
    print(f"Saved → {relative_path}/")

Spark version: 3.5.8
